# Chapter 9: Tool calling & MCP

We give an LLM four tools over ShopFlow (look up an order, find orders by email, check stock,
propose a refund), build the agent loop with the plain OpenAI SDK, then serve the **same** tools
over MCP (stdio and streamable HTTP) and run the same agent through an MCP client.

Run from this folder: `cd ch09/01_main-chapter-code && uv run jupyter lab`. Prerequisites:
ShopFlow DB running and seeded (`datasets/shopflow`), `.env` filled in, for the local-model rows, Ollama
running with `qwen3:8b` pulled. The local rows are off by default; run with `RUN_LOCAL=1` to include
them (`OLLAMA_BASE_URL` overrides `http://localhost:11434/v1`).

In [1]:
import json, os, socket, statistics, subprocess, sys, time
from pathlib import Path

CODE = Path.cwd()
sys.path.insert(0, str(CODE))

import ch09_agent, ch09_mcp_client, ch09_refunds, ch09_setup, ch09_tools
import ch09_scenarios as sc
from llmprod import chat_client, connect, get_settings

ch09_setup.setup_tables()   # idempotent: creates ch09_pending_refunds if missing
sc.reset_all()              # start from an empty refund queue
settings = get_settings()
MODEL = settings.llm_model
RUN_LOCAL = os.environ.get("RUN_LOCAL") == "1"   # local-model rows (Ollama) are opt-in
OLLAMA_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1")
print("model:", MODEL, "|", settings.llm_base_url, "| local rows:", RUN_LOCAL)
with connect() as conn:
    print("orders in DB:", conn.execute("select count(*) from orders").fetchone()[0])

model: openai/gpt-6-luna | https://openrouter.ai/api/v1 | local rows: True
orders in DB: 30433


## 1. What a tool call actually is

The model never runs anything. We describe each tool as JSON Schema, the model answers with a
**request** to call one (`tool_calls`), *we* run it, and we send the result back as a `tool`
message. Here is one round trip by hand, before any loop.

In [2]:
toolbox = ch09_tools.LocalToolbox()
print(json.dumps(toolbox.specs[0], indent=2))

{
  "type": "function",
  "function": {
    "name": "get_order",
    "description": "Look up one order: status, totals (integer cents), items with SKUs, payments, refunds, shipment dates and any pending refund requests. Does not return the customer's email.",
    "parameters": {
      "additionalProperties": false,
      "properties": {
        "order_id": {
          "description": "ShopFlow order id, a positive integer",
          "exclusiveMinimum": 0,
          "type": "integer"
        }
      },
      "required": [
        "order_id"
      ],
      "type": "object"
    }
  }
}


In [3]:
client = chat_client()
ctx0 = sc.pick_context(offset=0)
oid = ctx0["small"]["id"]
messages = [
    {"role": "system", "content": ch09_agent.SYSTEM_PROMPT},
    {"role": "user", "content": f"What's the status of order {oid}?"},
]
r1 = client.chat.completions.create(
    model=MODEL, messages=messages, tools=toolbox.specs, max_tokens=512, reasoning_effort="none"
)
choice = r1.choices[0]
print("finish_reason:", choice.finish_reason)
call = choice.message.tool_calls[0]
print("tool call    :", call.function.name, call.function.arguments)
print("usage        :", r1.usage.prompt_tokens, "prompt +", r1.usage.completion_tokens, "completion tokens")

finish_reason: tool_calls
tool call    : get_order {"order_id":2}
usage        : 683 prompt + 19 completion tokens


In [4]:
result = toolbox.call(call.function.name, call.function.arguments)   # WE run the tool
print(result[:300], "...")
messages += [
    {"role": "assistant", "content": "", "tool_calls": [
        {"id": call.id, "type": "function",
         "function": {"name": call.function.name, "arguments": call.function.arguments}}]},
    {"role": "tool", "tool_call_id": call.id, "content": result},
]
r2 = client.chat.completions.create(
    model=MODEL, messages=messages, tools=toolbox.specs, max_tokens=512, reasoning_effort="none"
)
print("\nfinal answer:", r2.choices[0].message.content)

{"id": 2, "status": "delivered", "channel": "web", "currency": "USD", "subtotal_cents": 8199, "discount_cents": 820, "shipping_cents": 599, "total_cents": 7978, "placed_at": "2026-05-13 11:47:20.975887-04:00", "customer_id": 921, "customer_name": "Emily Cox", "customer_segment": "consumer", "items": ...



final answer: Order 2 is **delivered**. It was delivered by DHL on May 16, 2026.


## 2. v0: the naive loop

The smallest loop that works: call the model, run whatever it asks, repeat until it stops asking.
It calls the Python functions directly and trusts the model's arguments.

In [5]:
def naive_agent(question):
    messages = [{"role": "system", "content": ch09_agent.SYSTEM_V0},
                {"role": "user", "content": question}]
    while True:                                           # no step limit
        msg = client.chat.completions.create(
            model=MODEL, messages=messages, tools=toolbox.specs, max_tokens=512,
            reasoning_effort="none").choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": [
            {"id": t.id, "type": "function",
             "function": {"name": t.function.name, "arguments": t.function.arguments}}
            for t in msg.tool_calls]})
        for t in msg.tool_calls:
            fn = ch09_tools.TOOL_FUNCTIONS[t.function.name]      # trusts the name
            out = fn(**json.loads(t.function.arguments))         # trusts the arguments
            messages.append({"role": "tool", "tool_call_id": t.id, "content": json.dumps(out, default=str)})

print(naive_agent(f"What's the status of order {oid}?"))

Order 2 is **delivered**. DHL marked it delivered on May 16, 2026. The $79.78 payment succeeded, and there are no refunds or pending refund requests.


In [6]:
try:
    print(naive_agent("What's the status of order 99999999?"))
except Exception as exc:
    print(type(exc).__name__, "->", exc)

ToolError -> order 99999999 not found


## 3. Measuring v0: what the hardened loop changes

The first user who mistypes an order id crashes v0. The fix is not a try/except in the loop; it is
a rule about *who the error is for*. A tool failure is information for the model, so it becomes a
`tool` message. `LocalToolbox.call` never raises. Each row below is a different way for a call to go
wrong, with the exact message the model receives.

In [7]:
bad_calls = [
    ("nonexistent order", "get_order", '{"order_id": 99999999}'),
    ("unknown tool", "delete_everything", "{}"),
    ("truncated JSON", "get_order", '{"order_id": 12'),
    ("wrong type", "get_order", '{"order_id": "twelve"}'),
    ("invented argument", "check_inventory", '{"sku": "SF-ELE-0001", "warehouse": "TX-01"}'),
    ("dollars instead of cents (negative)", "create_refund",
     '{"order_id": 2, "amount_cents": -25, "reason": "damaged", "idempotency_key": "k"}'),
    ("reason not in the enum", "create_refund",
     '{"order_id": 2, "amount_cents": 2500, "reason": "item broke", "idempotency_key": "k"}'),
    ("malformed email", "search_orders", '{"customer_email": "olivia"}'),
]
for label, name, raw in bad_calls:
    print(f"{label:38s} -> {toolbox.call(name, raw)}")

nonexistent order                      -> {"error": "order 99999999 not found"}
unknown tool                           -> {"error": "unknown tool 'delete_everything'. Available tools: ['check_inventory', 'create_refund', 'get_order', 'search_orders']"}
truncated JSON                         -> {"error": "arguments are not valid JSON: Expecting ',' delimiter (position 15)"}
wrong type                             -> {"error": "invalid arguments for get_order: order_id: Input should be a valid integer, unable to parse string as an integer"}
invented argument                      -> {"error": "invalid arguments for check_inventory: warehouse: Extra inputs are not permitted"}
dollars instead of cents (negative)    -> {"error": "invalid arguments for create_refund: amount_cents: Input should be greater than 0"}
reason not in the enum                 -> {"error": "invalid arguments for create_refund: reason: Input should be 'damaged', 'defective', 'wrong_item', 'not_delivered', 'late', 'chang

## 4. Does tool calling work on each provider?

OpenAI's model page for `gpt-6-luna` says Chat Completions supports function calling *only* with
`reasoning_effort` set to `none`. We test that through OpenRouter, with and without the parameter,
and on a second hosted model. The same question, three runs each. With `RUN_LOCAL=1`, `qwen3:8b`
on Ollama joins the comparison.

In [8]:
import openai

ollama = openai.OpenAI(base_url=OLLAMA_URL, api_key="ollama", timeout=900)
oid, sku = ctx0["small"]["id"], ctx0["stock"]["sku"]
Q = f"Is order {oid} delivered, and how many units of {sku} are in stock in total?"
want_total = int(ctx0["stock"]["total"])

CONFIGS = [
    ("openai/gpt-6-luna", "no reasoning_effort", None, client, 3),
    ("openai/gpt-6-luna", 'reasoning_effort="none"', {"reasoning_effort": "none"}, client, 3),
    ("openai/gpt-6-luna", 'reasoning_effort="low"', {"reasoning_effort": "low"}, client, 3),
    ("anthropic/claude-haiku-5.5", "no reasoning_effort", None, client, 3),
    ("anthropic/claude-haiku-5.5", 'reasoning_effort="none"', {"reasoning_effort": "none"}, client, 3),
]
if RUN_LOCAL:
    CONFIGS.append(("qwen3:8b (Ollama)", 'reasoning_effort="none"', {"reasoning_effort": "none"}, ollama, 3))
compat = []
for model, label, extra, cl, reps in CONFIGS:
    for _ in range(reps):
        name = model.split(" ")[0]
        res = ch09_agent.run_agent(Q, toolbox, client=cl, model=name, extra=extra, max_tokens=1024)
        used = {c.name for c in res.tool_calls}
        ok = {"get_order", "check_inventory"} <= used and str(want_total) in res.answer.replace(",", "")
        compat.append(dict(model=model, setting=label, ok=ok, stop=res.stop_reason, steps=res.steps,
                           parallel=res.max_parallel, max_prompt=res.max_prompt_tokens, tokens=res.total_tokens, cost=res.cost_usd,
                           ms=res.latency_ms, note="" if res.stop_reason == "done" else res.answer[:160]))
import pandas as pd
cdf = pd.DataFrame(compat)
summary = (cdf.groupby(["model", "setting"], sort=False)
    .agg(ok=("ok", "sum"), runs=("ok", "size"), max_parallel=("parallel", "max"), max_prompt=("max_prompt", "max"),
         tokens=("tokens", "mean"), cost_usd=("cost", "mean"), p50_ms=("ms", "median")))
summary["tokens"] = summary["tokens"].round(0).astype(int)
summary["cost_usd"] = summary["cost_usd"].round(5)
summary

ok  runs  max_parallel  \
model                      setting                                           
openai/gpt-6-luna          no reasoning_effort       3     3             2   
                           reasoning_effort="none"   3     3             2   
                           reasoning_effort="low"    3     3             2   
anthropic/claude-haiku-5.5 no reasoning_effort       3     3             2   
                           reasoning_effort="none"   3     3             2   
qwen3:8b (Ollama)          reasoning_effort="none"   3     3             2   

                                                    max_prompt  tokens  \
model                      setting                                       
openai/gpt-6-luna          no reasoning_effort            1116    1894   
                           reasoning_effort="none"        1116    1893   
                           reasoning_effort="low"         1116    1893   
anthropic/claude-haiku-5.5 no reasoning_effort            2166    3951   
                           reasoning_effort="none"        2175    3935   
qwen3:8b (Ollama)          reasoning_effort="none"        1346    2295   

                                                    cost_usd  p50_ms  
model                      setting                                    
openai/gpt-6-luna          no reasoning_effort       0.00016  2514.0  
                           reasoning_effort="none"   0.00016  3475.0  
                           reasoning_effort="low"    0.00016  3521.0  
anthropic/claude-haiku-5.5 no reasoning_effort       0.00049  2481.0  
                           reasoning_effort="none"   0.00048  2321.0  
qwen3:8b (Ollama)          reasoning_effort="none"   0.00000  2570.0

In [9]:
print(cdf[cdf.note != ""][["model", "setting", "stop", "note"]].to_string())

Empty DataFrame
Columns: [model, setting, stop, note]
Index: []


## 5. The scenario suite

Fourteen scenarios (listed in `ch09_scenarios.py`), each with a mechanical check on tool calls and
database state, never on how the answer reads. The configurations run concurrently, each on its own
set of orders: `gpt-6-luna` and `claude-haiku-5.5` (3 repetitions each), `qwen3:8b` on Ollama
(3, only with `RUN_LOCAL=1`),
and `gpt-6-luna` through the **MCP server** over stdio (1) and over streamable HTTP (1).
"direct" means the in-process `LocalToolbox`. We start the HTTP server here and stop it at the end of
the cell.

In [10]:
HTTP_URL = "http://127.0.0.1:8009/mcp"

def port_open(port):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

assert not port_open(8009), "port 8009 is already in use"
Path("results").mkdir(exist_ok=True)
log = open("results/mcp_http_server.log", "w")
server = subprocess.Popen(
    [sys.executable, "ch09_mcp_server.py", "--transport", "http", "--port", "8009"],
    cwd=CODE, stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    if port_open(8009):
        break
    time.sleep(0.5)
print("HTTP MCP server listening on :8009 ->", port_open(8009))

HTTP MCP server listening on :8009 -> True


In [11]:
from concurrent.futures import ThreadPoolExecutor

NONE = {"reasoning_effort": "none"}
suites = {}
try:
    with ch09_mcp_client.McpToolbox(ch09_mcp_client.stdio_server()) as mcp_stdio, \
         ch09_mcp_client.McpToolbox(HTTP_URL) as mcp_http:
        print("MCP protocol negotiated: stdio =", mcp_stdio.protocol_version, "| http =", mcp_http.protocol_version)
        jobs = {
            "luna / direct": ("openai/gpt-6-luna", toolbox, 0, 3, NONE, None),
            "haiku / direct": ("anthropic/claude-haiku-5.5", toolbox, 1, 3, None, None),
            "luna / MCP stdio": ("openai/gpt-6-luna", mcp_stdio, 3, 1, NONE, None),
            "luna / MCP http": ("openai/gpt-6-luna", mcp_http, 4, 1, NONE, None),
        }
        if RUN_LOCAL:
            jobs["qwen3:8b / direct"] = ("qwen3:8b", toolbox, 2, 3, NONE, ollama)
        t0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
            futs = {label: pool.submit(sc.run_suite, label, model, tb, offset=off, reps=reps,
                                       extra=extra, client=cl)
                    for label, (model, tb, off, reps, extra, cl) in jobs.items()}
            suites = {label: f.result() for label, f in futs.items()}
        print(f"all suites finished in {time.perf_counter() - t0:.0f}s")
finally:
    server.terminate()
    server.wait(timeout=15)
    log.close()
print("HTTP server stopped; port 8009 open:", port_open(8009))
server_log = Path("results/mcp_http_server.log")
print("HTTP requests served:", server_log.read_text().count("POST /mcp"), "POST /mcp,",
      server_log.read_text().count("GET /mcp"), "GET /mcp")
server_log.unlink()

MCP protocol negotiated: stdio = 2026-07-28 | http = 2026-07-28


all suites finished in 195s


HTTP server stopped; port 8009 open: False
HTTP requests served: 25 POST /mcp, 0 GET /mcp


In [12]:
summ = pd.DataFrame([sc.summarize(s) for s in suites.values()]).set_index("label")
summ["pass_rate"] = (100 * summ["pass"] / summ["total"]).round(1)
summ["cost_usd"] = summ["cost_usd"].round(4)
summ[["pass", "total", "pass_rate", "tool_errors", "tokens_per_run", "max_prompt_tokens", "p50_ms", "p95_ms", "cost_usd"]]

,pass,total,pass_rate,tool_errors,tokens_per_run,max_prompt_tokens,p50_ms,p95_ms,cost_usd
label,,,,,,,,,
luna / direct,42,42,100.0,6,2363,1352,4065,6250,0.0099
haiku / direct,42,42,100.0,3,4859,2462,3059,4568,0.0263
luna / MCP stdio,14,14,100.0,2,2465,1371,4193,5931,0.0039
luna / MCP http,14,14,100.0,2,2446,1272,4279,9673,0.0040
qwen3:8b / direct,36,42,85.7,9,3070,1758,3524,6389,0.0000


In [13]:
ids = [s.id for s in sc.SCENARIOS]
matrix = {}
for label, s in suites.items():
    col = {}
    for i in ids:
        runs = [r for r in s["runs"] if r["id"] == i]
        col[i] = f"{sum(r['passed'] for r in runs)}/{len(runs)}"
    matrix[label] = col
pd.DataFrame(matrix)

,luna / direct,haiku / direct,luna / MCP stdio,luna / MCP http,qwen3:8b / direct
S1,3/3,3/3,1/1,1/1,3/3
S2,3/3,3/3,1/1,1/1,3/3
S3,3/3,3/3,1/1,1/1,3/3
S4,3/3,3/3,1/1,1/1,3/3
S5,3/3,3/3,1/1,1/1,0/3
S6,3/3,3/3,1/1,1/1,3/3
S7,3/3,3/3,1/1,1/1,3/3
S8,3/3,3/3,1/1,1/1,3/3
S9,3/3,3/3,1/1,1/1,3/3
S10,3/3,3/3,1/1,1/1,3/3


In [14]:
fails = [(label, r["rep"], r["id"], r["detail"], r["tool_calls"], r["answer"][:200].replace("\n", " "))
         for label, s in suites.items() for r in s["runs"] if not r["passed"]]
print(len(fails), "failing runs")
for f in fails:
    print(f)

6 failing runs
('qwen3:8b / direct', 1, 'S5', 'get_order at step 1, check_inventory(SF-FOO-0148) at step -', ['get_order'], 'The product in order 6 is the "Premium Slip-On Loafer" (SKU: SF-FOO-0148). To check if it\'s in stock, we need to look up its inventory. Let me do that for you.')
('qwen3:8b / direct', 1, 'S13', 'create_refund calls=1, rows=1, checked order=True', ['get_order', 'create_refund'], 'The ticket mentions order #8, which is marked as delivered. The customer claims the parcel is late, but the system shows it was delivered on May 15, 2026.  A refund request for the full amount ($287.0')
('qwen3:8b / direct', 2, 'S5', 'get_order at step 1, check_inventory(SF-FOO-0148) at step -', ['get_order'], 'The product in order 6 is the "Premium Slip-On Loafer" with SKU `SF-FOO-0148`. To check if it\'s in stock, I\'ll look up the inventory for this SKU.')
('qwen3:8b / direct', 2, 'S13', 'create_refund calls=1, rows=1, checked order=True', ['get_order', 'create_refund'], 'The ticket m

In [15]:
rows = []
for label, s in suites.items():
    for r in s["runs"]:
        rows.append({"label": label, **{k: r[k] for k in ("id", "passed", "steps", "max_parallel", "tool_errors", "tokens", "latency_ms")}})
runs_df = pd.DataFrame(rows)
print("runs that issued >=2 tool calls in one assistant turn:",
      int((runs_df.max_parallel >= 2).sum()), "of", len(runs_df))
print("tool errors seen by the model (all suites):", int(runs_df.tool_errors.sum()))
runs_df[runs_df.tool_errors > 0].groupby(["label", "id"]).tool_errors.sum()

runs that issued >=2 tool calls in one assistant turn: 11 of 154
tool errors seen by the model (all suites): 22


label              id 
haiku / direct     S10    3
luna / MCP http    S10    1
                   S12    1
luna / MCP stdio   S10    1
                   S12    1
luna / direct      S10    3
                   S12    3
qwen3:8b / direct  S10    3
                   S11    3
                   S12    3
Name: tool_errors, dtype: int64

In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
palette = {"luna / direct": "#4a7fd1", "haiku / direct": "#8a5cc9", "qwen3:8b / direct": "#c9961c"}
colors = [palette.get(label, "#3f9a5b") for label in summ.index]   # MCP rows in green
ax[0].bar(summ.index, summ["pass_rate"], color=colors)
ax[0].set_ylim(0, 105); ax[0].set_ylabel("scenario pass rate (%)")
ax[0].set_title("14 scenarios, mechanical checks")
for i, v in enumerate(summ["pass_rate"]):
    ax[0].text(i, v + 1.5, f"{v:.0f}%", ha="center", fontsize=9)
ax[1].bar(summ.index, summ["p50_ms"] / 1000, color=colors)
ax[1].set_ylabel("p50 seconds per conversation"); ax[1].set_title("latency (median)")
for i, v in enumerate(summ["p50_ms"] / 1000):
    ax[1].text(i, v + 0.1, f"{v:.1f}s", ha="center", fontsize=9)
for a in ax:
    a.tick_params(axis="x", labelrotation=25)
plt.tight_layout()
Path("../images").mkdir(exist_ok=True)
plt.savefig("../images/09-scenarios.png", dpi=130)
print("saved ../images/09-scenarios.png")

saved ../images/09-scenarios.png


## 6. Hard stops with a live model

`max_steps` and `token_budget` exist for the day a model loops or a tool returns a novel. We trigger
both on purpose with a real model and a question that needs three tool calls. The budget check
projects the next call before making it: tokens spent so far + the last prompt (the conversation
only grows) + the tool results appended since (~4 characters per token). With 1,500 tokens the
tools run but the call that would read them is never made; with 1,000 tokens the requested tools
are not even executed.

In [17]:
q = (f"Compare orders {ctx0['pair']['a_id']} and {ctx0['pair']['b_id']}, then check the stock of "
     f"{ctx0['single']['sku']} and tell me which order has more items.")
for kwargs in ({"max_steps": 8, "token_budget": 20_000}, {"max_steps": 1}, {"token_budget": 1_500}, {"token_budget": 1_000}):
    r = ch09_agent.run_agent(q, toolbox, client=client, model=MODEL, extra=NONE, **kwargs)
    print(kwargs, "->", r.stop_reason, "| steps", r.steps, "| tool calls run", len(r.tool_calls),
          "| tokens", r.total_tokens)
    print("   answer:", r.answer[:150].replace("\n", " "))
    if r.stop_reason != "done":
        print("   last message:", r.messages[-1]["content"])

{'max_steps': 8, 'token_budget': 20000} -> done | steps 2 | tool calls run 3 | tokens 2374
   answer: - **Order 1:** 3 items total, including 1 Everyday USB-C Hub (SF-ELE-0073); total **$289.97**. - **Order 6569:** 3 Pro Slip-On Loafers; total **$311.9


{'max_steps': 1} -> max_steps | steps 1 | tool calls run 0 | tokens 774
   answer: Stopped without a final answer: max steps reached.
   last message: {"error": "not executed: the run hit its step/token limit"}


{'token_budget': 1500} -> token_budget | steps 1 | tool calls run 3 | tokens 774
   answer: Stopped without a final answer: token budget reached.
   last message: {"sku": "SF-ELE-0073", "name": "Everyday USB-C Hub", "price_cents": 8199, "active": true, "warehouses": [{"warehouse": "CA-01", "on_hand": 480}, {"warehouse": "NJ-01", "on_hand": 0}, {"warehouse": "TX-01", "on_hand": 377}], "total_on_hand": 857}


{'token_budget': 1000} -> token_budget | steps 1 | tool calls run 0 | tokens 774
   answer: Stopped without a final answer: token budget reached.
   last message: {"error": "not executed: the run hit its step/token limit"}


## 7. The human side: `approve.py`

The model can only *propose*. We have the agent queue two refunds (one within the agent limit, one
over it), then act as different humans with the CLI.

In [18]:
ctx7 = sc.pick_context(offset=7)
sc.clear_refunds([ctx7["small"]["id"], ctx7["mid"]["id"]])
for s_ in (sc.SCENARIOS[5], sc.SCENARIOS[6]):            # S6 within limit, S7 over limit
    r = ch09_agent.run_agent(s_.question(ctx7), toolbox, client=client, model=MODEL, extra=NONE)
    print(s_.id, "->", r.answer[:170].replace("\n", " "))

def cli(*args):
    p = subprocess.run([sys.executable, "approve.py", *args], cwd=CODE, capture_output=True, text=True)
    print("$ approve.py", " ".join(args), f"[exit {p.returncode}]")
    print((p.stdout + p.stderr).rstrip())

cli("list")

S6 -> The $25.00 refund for order 53 was submitted for approval by an agent. It has not been refunded yet.


S7 -> The full refund request for **$245.97** on order 34 was submitted for approval by a **team lead**. It has not been refunded yet.


$ approve.py list [exit 0]
 id   order      amount  reason         status    needs     key
 48      53      $25.00  damaged        pending   agent     refund-53-damaged
 49      34     $245.97  defective      pending   team_lead refund-34-defective


In [19]:
rows = [r for r in ch09_refunds.list_requests("pending") if r["order_id"] in (ctx7["small"]["id"], ctx7["mid"]["id"])]
lead_id = next(r["id"] for r in rows if r["required_approver"] == "team_lead")
agent_id = next(r["id"] for r in rows if r["required_approver"] == "agent")
cli("approve", str(lead_id), "--role", "agent")                       # too junior: refused
cli("approve", str(lead_id), "--role", "team_lead", "--note", "defect confirmed")
cli("reject", str(agent_id), "--role", "agent", "--note", "outside return window")
cli("approve", str(lead_id), "--role", "finance")                    # already decided
cli("list", "--status", "all")

$ approve.py approve 49 --role agent [exit 1]
refused: role 'agent' cannot approve $245.97: requires 'team_lead' or higher


$ approve.py approve 49 --role team_lead --note defect confirmed [exit 0]
request 49 approved by team_lead: $245.97 on order 34


$ approve.py reject 48 --role agent --note outside return window [exit 0]
request 48 rejected by agent: $25.00 on order 53


$ approve.py approve 49 --role finance [exit 1]
refused: refund request 49 is already approved


$ approve.py list --status all [exit 0]
 id   order      amount  reason         status    needs     key
 48      53      $25.00  damaged        rejected  agent     refund-53-damaged
 49      34     $245.97  defective      approved  team_lead refund-34-defective


## 8. MCP: the same tools, a different door

`ch09_mcp_server.py` registers the very functions in `ch09_tools.py`. We check what a client sees
over stdio: protocol version, tool list, the safety hints, and that the converted schemas match the
ones the raw loop used.

In [20]:
from mcp.client.client import Client

async with Client(ch09_mcp_client.stdio_server()) as mc:
    print("server :", mc.server_info.name, mc.server_info.version, "| protocol", mc.protocol_version)
    tools = (await mc.list_tools()).tools
    for t in tools:
        a = t.annotations
        print(f"  {t.name:16s} read_only={a.read_only_hint!s:5} idempotent={a.idempotent_hint}")
    converted = [ch09_mcp_client.mcp_tool_to_openai(t) for t in tools]

local = {s["function"]["name"]: s["function"]["parameters"] for s in toolbox.specs}
for c in converted:
    n = c["function"]["name"]
    same = (set(c["function"]["parameters"]["properties"]) == set(local[n]["properties"])
            and set(c["function"]["parameters"].get("required", [])) == set(local[n].get("required", [])))
    print(f"schema parity for {n}: {same}")
print(json.dumps(converted[3]["function"]["parameters"]["properties"]["reason"]))

server : shopflow-support 0.1.0 | protocol 2026-07-28
  get_order        read_only=True  idempotent=None
  search_orders    read_only=True  idempotent=None
  check_inventory  read_only=True  idempotent=None
  create_refund    read_only=False idempotent=True
schema parity for get_order: True
schema parity for search_orders: True
schema parity for check_inventory: True
schema parity for create_refund: True
{"description": "Reason code from the allowed list", "enum": ["damaged", "defective", "wrong_item", "not_delivered", "late", "changed_mind", "other"], "type": "string"}


In [21]:
with ch09_mcp_client.McpToolbox(ch09_mcp_client.stdio_server()) as tb:
    print(tb.call("get_order", '{"order_id": 99999999}'))
    print(tb.call("create_refund", '{"order_id": 2, "amount_cents": 0, "reason": "late", "idempotency_key": "k"}')[:230])

{"error": "Error executing tool get_order: order 99999999 not found"}
{"error": "Error executing tool create_refund: 1 validation error for create_refundArguments\namount_cents\n  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]\n    For further information visit htt


In [22]:
sc.reset_all()
out = Path("results"); out.mkdir(exist_ok=True)
(out / "scenario_suites.json").write_text(json.dumps(
    {"compat": compat, "suites": suites, "summary": json.loads(summ.reset_index().to_json(orient="records"))},
    indent=1, default=str))
print("saved results/scenario_suites.json")

saved results/scenario_suites.json
